🌐 [English](dx_stream.ipynb) | [한국어](dx_stream.ko.ipynb) | **日本語** | [中文](dx_stream.zh.ipynb)

# DEEPX Tutorial 04 - DX-STREAM ワークフロー

このチュートリアルでは DX-STREAM v3.1.x を紹介し、DEEPX NPU 上でエンドツーエンドの Vision AI パイプラインを構築する方法を説明します。

## 学習目標

このチュートリアルでは次のことを学びます。

- DEEPX SDK における DX-STREAM の位置づけを理解する
- `DxPreprocess → DxInfer → DxPostprocess → DxTracker → DxOsd` でパイプラインを構築する
- DX マルチストリームドメインとセレクターの境界を学ぶ
- 推論バックエンド、スケーリング、変換、メッセージブローカーの各エレメントを使う
- カスタム AI モデル向けにカスタム後処理ライブラリを適応させる
- メタデータ駆動の人数カウントオーバーレイを追加する
- メタデータを調べ、GStreamer パイプラインを診断する

## 1. DX-STREAM の概要

DX-STREAM は、DEEPX NPU 上で Vision AI パイプラインを構築するための GStreamer エレメント群です。

```text
Source → Decode → DxPreprocess → DxInfer → DxPostprocess → DxTracker → DxOsd → Sink
```

![DX-STREAM パイプライン](assets/dx-stream-pipeline.png)

| エレメント | 用途 |
|---|---|
| `dxpreprocess` | フレームをリサイズ、クロップし、モデル入力テンソルに変換します |
| `dxinfer` | コンパイル済みの `.dxnn` モデルを実行します |
| `dxpostprocess` | 出力テンソルをデコードし、推論メタデータを書き込みます |
| `dxtracker` | 検出されたオブジェクトに永続的な ID を割り当てます |
| `dxosd` | ボックス、ラベル、姿勢、セグメンテーション結果を描画します |
| `dxgather` | 同じソースから作成されたブランチを統合します |
| `dxinputselector` | 入力ストリームを DX マルチストリームドメインに統合します |
| `dxoutputselector` | DX マルチストリームドメインをストリーム ID ごとに分割します |
| `dxrate` | 出力フレームレートを制御します |
| `dxmsgconv` | 推論メタデータを構造化メッセージに変換します |
| `dxmsgbroker` | メッセージを MQTT または Kafka に配信します |
| `dxscale` | フレームの解像度を変更します |
| `dxconvert` | フレームのカラーフォーマットを変更します |

エレメントとプロパティの完全なリファレンスは、[DEEPX Developer Portal](https://developer.deepx.ai/download/?id=583) から入手できる DX-STREAM User Manual を参照してください。ログインが必要です。

## 2. 前提条件

- チュートリアル 01 を `./dx-runtime/install.sh --all` で完了していること: DX-RT、DX-APP、DX-STREAM がビルド済みで、NPU が認識されていること。チュートリアル 03 の完了を推奨します。第 5 節では、そこで作成した Forklift and Worker モデルが存在すれば再利用します。
- ウィンドウ表示のためのグラフィカルデスクトップセッション。セットアップセルはディスプレイを検出します。ディスプレイがあればパイプラインのセルは DX-STREAM ウィンドウを開き、ない場合(たとえば SSH 経由)は `fakesink` を使って**ヘッドレス**で実行し、ノートブックが最後まで完了するようにします。`HEADLESS` はどちらの値にも強制設定できます。第 4 節の `run_demo.sh` デモとカメラのセルはそれぞれ数分かかるため、オプトイン(`RUN_DEMOS`、`RUN_CAMERA`)になっています。
- 任意のハードウェア: 4.3 用の V4L2 カメラ(`RUN_CAMERA`)、4.9 用の到達可能な RTSP ストリーム、4.11 用の MQTT または Kafka ブローカー。
- ツール: `gstreamer1.0-tools`、`v4l-utils`、`jq`(`setup.sh` が使用)、7.3 用の `graphviz`。5.7 節では `meson` で DX-STREAM を再ビルドするため、`sudo` が必要です。
- DX-STREAM GStreamer プラグインが `gst-inspect-1.0` から見えていること。`build.sh` は `export GST_PLUGIN_PATH=...` の行を表示します。セットアップセルはこれを確認し、プラグインが既定の場所にある場合はカーネル用にこの変数を設定します。
- ダウンロード: `setup.sh` はサンプルモデル 17 個(約 300 MB)と共有サンプル動画アーカイブ(約 1.1 GB。`<dx-all-suite>/workspace/res/videos` に保存され、DX-APP と共有されます)を取得します。第 5 節はチュートリアル 03 のモデルと動画を再利用するか、90 MB をダウンロードします。
- このチュートリアルで作成されるファイルは `notebooks/T04-DX-STREAM/workspace/` に置かれ、git では無視されます。SDK ツリー内で変更されるのは 5.6 の後処理パッチだけで、それがその節の目的です。

次のセルは SDK の場所を特定し、これらの要件を確認してステータス表を表示します。

### 2.1 SDK パスの読み込み

次のセルは `DX_ALL_SUITE_DIR` から `DX_STREAM_DIR` とその他の SDK パスを導出し、ノートブックの作業ディレクトリを、`run_demo.sh` とサンプルのパスが前提とする DX-STREAM リポジトリのルートに変更します。また、以降で使用する表示フラグとワークスペースのパスも定義します。

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial

import os, subprocess

paths = setup_tutorial(needs=["dx_rt", "dx_stream", "gst_dxstream", "npu"])
paths.require("dx_rt", "dx_stream", "gst_dxstream", "npu")   # stop here with the fix if a prerequisite is missing
%cd {DX_STREAM_DIR}

####################################################################
HAS_DISPLAY = bool(os.environ.get("DISPLAY") or os.environ.get("WAYLAND_DISPLAY"))
HEADLESS = not HAS_DISPLAY   # a display was detected: open the DX-STREAM window. No display (e.g. SSH): fakesink. Set True/False to force it.
RUN_DEMOS = True    # run_demo.sh demos always open a window; set True on a desktop session to run them from the cells.
RUN_CAMERA = True   # the two camera pipelines in 4.3
####################################################################
# videoconvert (software) before the display sink: the hardware dxconvert produced intermittent all-green frames on some clips.
SINK = "fakesink sync=false" if HEADLESS else "videoconvert ! fpsdisplaysink sync=false"
DISPLAY_FLAG = "--no-display" if HEADLESS else ""

# Files created by this tutorial (ignored by git). Section 5 reuses Tutorial 03 outputs when they exist.
WORKSPACE_DIR.mkdir(exist_ok=True)
T03_WORKSPACE = TUTORIAL_ROOT / "notebooks" / "T03-E2E-AI-Workflow" / "workspace"
CUSTOM_DXNN = WORKSPACE_DIR / "yolov7-forklift-person.dxnn"
CUSTOM_VIDEO = WORKSPACE_DIR / "forklift-worker.mp4"
CUSTOM_JSON = WORKSPACE_DIR / "yolov7-forklift-person.json"
PATCH_FILE = WORKSPACE_DIR / "dx_stream_update.diff"
APP_PATH = WORKSPACE_DIR / "yolo26_people_counter.py"
DOT_DIR = WORKSPACE_DIR / "dot"
YOLO26S_MODEL = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"     # downloaded by Tutorial 01 section 3.3
PEOPLE_VIDEO = DX_STREAM_DIR / "dx_stream" / "samples" / "videos" / "snowboard.mp4"
DX_STREAM_PYTHON = DX_STREAM_DIR / "venv-dx_stream" / "bin" / "python"             # created by build.sh; has pydxs and gi
print()
print(f"Display detected: {HAS_DISPLAY}  ->  HEADLESS={HEADLESS}  (sink: {SINK})   RUN_DEMOS={RUN_DEMOS}  RUN_CAMERA={RUN_CAMERA}")
print(f"Workspace: {WORKSPACE_DIR}")


### 2.2 (任意) Intel GPU の前提条件

Intel x86_64 iGPU/dGPU システムの場合のみ、[`docs/intel_gpu.md`](../../docs/intel_gpu.md) に従ってください。

### 2.3 サンプルリソースのダウンロード

`setup.sh` は、同梱パイプラインに必要なモデルと動画を `dx_stream/samples/` にダウンロードします。現在のリストにはモデル 17 個(約 300 MB)と共有サンプル動画アーカイブ(約 1.1 GB。`setup.sh` が `<dx-all-suite>/workspace/res/videos` に展開し、`dx_stream/samples/videos` としてリンクします)が含まれます。初回実行では数分かかります。サンプルがすでに存在する場合、セルはスクリプトをスキップします。

In [ ]:
# Download sample models and videos (skipped when they are already present).
SAMPLE_MODELS = DX_STREAM_DIR / "dx_stream" / "samples" / "models"
if len(list(SAMPLE_MODELS.glob("*.dxnn"))) >= 17 and PEOPLE_VIDEO.is_file():
    print(f"Samples are already present under {SAMPLE_MODELS.parent}; skipping ./setup.sh")
else:
    !./setup.sh


### 2.4 インストール済み DX-STREAM エレメントの確認

プラグインは次の 13 個のエレメントを公開しているはずです。

`dxconvert`、`dxgather`、`dxinfer`、`dxinputselector`、`dxmsgbroker`、`dxmsgconv`、`dxosd`、`dxoutputselector`、`dxpostprocess`、`dxpreprocess`、`dxrate`、`dxscale`、`dxtracker`。

In [ ]:
!gst-inspect-1.0 dxstream || echo "DX-STREAM plugin not found"


## 3. クイックスタート

### 3.1 YOLO26n 物体検出パイプラインの実行

このパイプラインは動画を読み込み、各フレームを前処理して YOLO26n を実行し、結果をデコードしてアノテーション付きの出力をレンダリングします。デスクトップセッション(`HEADLESS = False`、ディスプレイが検出された場合の既定値)では結果がウィンドウに表示され、動画が終了するか停止ボタン(■)を押すまでセルは実行中のままになります。ディスプレイがない場合(`HEADLESS = True`)、フレームは `fakesink` に送られ、セルはパイプラインの状態メッセージだけを表示します。この場合の目的は、すべてのエレメントがリンクされ、モデルが読み込まれ、推論が実行されることを確認することです。セルは使用するシンクを表示します。

代わりにターミナル(File > New > Terminal)で実行するには、セットアップセルが表示した DX-STREAM ディレクトリに `cd` し、下の `gst-launch-1.0` コマンドの最後のエレメントを `videoconvert ! fpsdisplaysink sync=false` にして貼り付けます。`Ctrl+C` で停止します。

In [ ]:
VIDEO_SRC = "dx_stream/samples/videos/doughnut.mp4"
MODEL_PATH = "dx_stream/samples/models/yolo26-n_640x640.dxnn"

print(f"Sink: {SINK}   (HEADLESS={HEADLESS}; set HEADLESS = False in the setup cell and rerun it to open a window)\n")
!gst-launch-1.0 filesrc location="{VIDEO_SRC}" ! decodebin ! \
    dxpreprocess \
        preprocess-id=1 \
        resize-width=640 \
        resize-height=640 ! \
    queue max-size-buffers=1 ! \
    dxinfer \
        preprocess-id=1 \
        inference-id=1 \
        model-path="{MODEL_PATH}" \
        backend=auto ! \
    queue max-size-buffers=1 ! \
    dxpostprocess \
        inference-id=1 \
        library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolo26od.so \
        function-name=PostProcess ! \
    queue max-size-buffers=1 ! \
    dxosd ! {SINK}


### 3.2 コアエレメントの確認

`gst-inspect-1.0` は、インストールされているバージョンのパッドテンプレート、プロパティ、既定値、サポートされる列挙値を表示します。

#### 3.2.1 DxPreprocess

```bash
dxpreprocess preprocess-id=1 resize-width=640 resize-height=640
```

`preprocess-id` は生成される入力テンソルを識別します。下流の `dxinfer` は同じ ID を参照する必要があります。

In [ ]:
!gst-inspect-1.0 dxpreprocess


#### 3.2.2 DxInfer と推論バックエンド

```bash
dxinfer preprocess-id=1 inference-id=1 model-path=/path/to/model.dxnn backend=auto
```

- `auto`: 利用可能なコンパイル済みバックエンドを選択します
- `dxrt`: DEEPX Runtime バックエンドを使用します
- `dxvnpu`: DX-STREAM が VNPU サポート付きでビルドされている場合に VNPU バックエンドを使用します

`inference-id` は `dxpostprocess` が消費する出力テンソルを識別します。DX-STREAM v3.1.x は内部で非同期の Put/Get バックエンドインターフェースを使用しますが、パイプラインの構文は変わりません。

In [ ]:
!gst-inspect-1.0 dxinfer


#### 3.2.3 DxPostprocess

```bash
dxpostprocess inference-id=1 \
  library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolo26od.so \
  function-name=PostProcess
```

`inference-id` は上流の推論出力と一致している必要があります。共有ライブラリと関数はモデルアーキテクチャに対応したものでなければなりません。

In [ ]:
!gst-inspect-1.0 dxpostprocess


#### 3.2.4 DxOsd

DxOsd は推論メタデータを読み取り、視覚的な結果を映像フレームに重ねて描画します。

In [ ]:
!gst-inspect-1.0 dxosd


### 3.3 映像フレームのスケーリングと変換

`dxscale` は解像度を変更し、`dxconvert` はカラーフォーマットを変更します。対応プラットフォームでは高速化されたカーネルが自動的に選択され、サポートされない組み合わせはソフトウェア実装にフォールバックします。

```bash
... ! dxscale width=640 height=480 ! \
      dxconvert ! video/x-raw,format=RGB ! ...
```

`dxconvert` はフレームをリサイズせず、`dxscale` は最終的なカラーフォーマットを選択しません。

このノートブックのパイプラインは、表示シンクの前にソフトウェアの `videoconvert` を置いて終わります。ハードウェアの `dxconvert` の方が高速ですが、この SDK リリースでは一部のクリップ(たとえば第 5 節の Forklift and Worker 動画)で断続的に全面緑色のフレームが生じました。同梱スクリプトは `VIDEOCONVERT_PIPELINE` 変数でプラットフォームごとにコンバーターを選択します。`SINK` を `dxconvert` に切り替えて緑色のフレームが表示される場合は、元に戻してください。

In [ ]:
!gst-inspect-1.0 dxscale


In [ ]:
!gst-inspect-1.0 dxconvert


## 4. 同梱デモパイプラインの実行

`run_demo.sh` には 12 の選択肢があります。`0` から `9`、Secondary Mode の `-`、Depth Estimation の `=` です。10 秒以内に選択がなければオプション `0` を実行します。次のセルはインストールされているメニューを表示します。

これらのデモはネイティブウィンドウを開き、パイプラインが終了するまでセルを占有するため、以下のセルはデスクトップセッションで `RUN_DEMOS = True`(セットアップセル)のときだけ実行されます。推奨される方法はターミナル(File > New > Terminal)です。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_stream     # the setup cell printed your actual path
./run_demo.sh
```

メニュー項目はすぐに入力し(プロンプトは 10 秒でタイムアウトします)、`Ctrl+C` でパイプラインを停止します。各デモの後には、実行されるスクリプトと、その中で注目すべき点についてのメモが続きます。

In [ ]:
# Display the menu implemented by the installed run_demo.sh.
!sed -n '/echo "0:/,/read -t/p' run_demo.sh


実行中のパイプラインはノートブックの停止ボタン(`■`)で停止します。

### 4.1 物体検出 - YOLO26n

![単一の物体検出パイプライン](assets/pipline-single-detection.png)

注目ポイント: クイックスタートと同じ 5 つの DX エレメント。`model-path` と `library-file-path=.../libpostprocess_yolo26od.so` が `dxinfer` と `dxpostprocess` に直接設定されています。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 0
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 0.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/object_detection/run_yolo26n.sh


### 4.2 PPU を使った物体検出 - YOLOv5s

注目ポイント: `library-file-path` がありません。各 DX エレメントは `dx_stream/configs/YoloV5S_PPU/` 配下の JSON `config-file-path` を読み込みます。PPU とは NPU がすでにボックスをデコード済みであることを意味し、後処理の設定は PPU 出力のマッピングだけを行います。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 1
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 1.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/object_detection/run_YoloV5S_PPU.sh


### 4.3 顔検出

注目ポイント: 4.1 の構造に顔モデルと `libpostprocess_yolov5s_face.so` を組み合わせたもの。`dxosd` が顔のボックスとランドマークを描画します。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 2
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 2.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/face_detection/run_YOLOv5s_Face.sh


次の例は PPU 出力付きの SCRFD500M モデルを使用します。

注目ポイント: 汎用の `libpostprocess_ppu.so`。PPU モデルでは、1 つの共有ライブラリが複数のアーキテクチャに対応します。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 3
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 3.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/face_detection/run_SCRFD500M_PPU.sh


#### カメラソースの使用

最初の例は raw 映像を要求します。2 つ目は MJPEG を要求しますが、すべての USB カメラがサポートしているわけではありません。`v4l2-ctl --list-formats-ext --device=/dev/video0` の結果に合わせてデバイスと caps を調整してください。

In [ ]:
# Camera input using a raw video format.
if RUN_CAMERA:
    !gst-launch-1.0 \
      v4l2src device=/dev/video0 do-timestamp=true ! \
      videoconvert ! video/x-raw,width=1920,height=1080 ! queue ! \
      dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
      queue max-size-buffers=1 ! \
      dxinfer \
        preprocess-id=1 inference-id=1 \
        model-path=dx_stream/samples/models/yolov5-s-face_640x640.dxnn \
        backend=auto ! \
      queue max-size-buffers=1 ! \
      dxpostprocess \
        inference-id=1 \
        library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolov5s_face.so \
        function-name=PostProcess ! \
      queue max-size-buffers=1 ! \
      dxosd ! videoconvert ! fpsdisplaysink sync=false
else:
    print("Skipped: set RUN_CAMERA = True in the setup cell when a camera and a display are available.")


In [ ]:
# Camera input using MJPEG.
if RUN_CAMERA:
    !gst-launch-1.0 \
      v4l2src device=/dev/video0 do-timestamp=true ! \
      image/jpeg,width=1920,height=1080,framerate=30/1 ! \
      jpegdec ! videoconvert ! queue ! \
      dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
      queue max-size-buffers=1 ! \
      dxinfer \
        preprocess-id=1 inference-id=1 \
        model-path=dx_stream/samples/models/yolov5-s-face_640x640.dxnn \
        backend=auto ! \
      queue max-size-buffers=1 ! \
      dxpostprocess \
        inference-id=1 \
        library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolov5s_face.so \
        function-name=PostProcess ! \
      queue max-size-buffers=1 ! \
      dxosd ! videoconvert ! fpsdisplaysink sync=false
else:
    print("Skipped: set RUN_CAMERA = True in the setup cell when a camera and a display are available.")


### 4.4 姿勢推定

注目ポイント: `libpostprocess_yolo26pose.so` がキーポイントを `DXObjectMeta` に書き込み、`dxosd` がそれをもとにスケルトンを描画します。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 4
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 4.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/pose_estimation/run_yolo26n-pose.sh


次の例は PPU 出力付きの YOLOv5 Pose を使用します。

注目ポイント: ここでも `libpostprocess_ppu.so` です。4.4 と比較してください。変わるのはモデルとライブラリだけです。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 5
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 5.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/pose_estimation/run_YOLOV5Pose_PPU.sh


### 4.5 インスタンスセグメンテーション - YOLO26n-Seg

オプション `6` はセグメンテーションデモを実行します。現在の SDK レイアウトでは、スクリプトは `instance_segmentation` 配下にあります。

注目ポイント: `dxpostprocess` と `dxosd` の間にある `dxtracker` エレメント。セグメンテーションデモはマスクにもトラック ID を割り当てます。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 6
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 6.")


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/instance_segmentation/run_yolo26n-seg.sh


### 4.6 複数物体追跡

オプション `7` は物体検出に続けて追跡を実行します。

![単一の追跡パイプライン](assets/pipline-single-tracking.png)

注目ポイント: 4.2 の検出チェーンに `dxtracker config-file-path=.../tracker_config.json` を加えたもの。トラッカーの設定でアルゴリズムとそのしきい値を選択します。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 7
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 7.")


In [ ]:
!sed -n '1,260p' dx_stream/pipelines/tracking/run_multi_object_tracker.sh


### 4.7 マルチチャネル物体検出

オプション `8` は 4 つの独立した推論ブランチを実行し、その出力を合成します。

![マルチチャネルパイプライン](assets/pipeline-multi-stream.png)

注目ポイント: 4 つの `filesrc ... ! dxscale` ブランチがそれぞれ独自の前処理、推論、後処理を持ち、`compositor` で統合されます。モデルはブランチごとに 1 回ずつ読み込まれます。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 8
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 8.")


In [ ]:
!sed -n '1,280p' dx_stream/pipelines/multi_stream/run_multi_stream.sh


### 4.8 DX マルチストリームドメイン

DX-STREAM v3.1.0 では `application/x-dxvideoraw` caps ドメインが導入されました。これにより、各ストリームの識別情報、寸法、フォーマット、メタデータ、タイムラインを保ったまま、複数のストリームが 1 つの処理チェーンを共有できます。

```text
N × video/x-raw
       ↓
dxinputselector                 domain entry
       ↓ application/x-dxvideoraw
dxpreprocess → dxinfer → dxpostprocess → dxosd
       ↓ application/x-dxvideoraw
dxoutputselector                domain exit
       ↓
N × video/x-raw
```

`dxinputselector` はストリーム ID を割り当て、必要に応じて `DXFrameMeta` を作成します。`dxoutputselector` は各バッファをルーティングし、対応するストリームごとのイベントを復元します。

このドメインはストリームごとの `STREAM_START`、`CAPS`、`SEGMENT`、`TAG`、`EOS`、`GAP` イベントを保持します。そのため、遅いストリームはすべての入力を抑制することなく、自身の QoS を独自に処理できます。

#### エレメント配置のルール

| エレメント | `application/x-dxvideoraw` 内 | 配置に関する注意 |
|---|---:|---|
| `dxpreprocess`、`dxinfer`、`dxpostprocess`、`dxtracker`、`dxosd`、`dxrate` | 可 | これらのエレメントは単一ストリームモードでもドメインモードでも動作します |
| `dxscale`、`dxconvert` | 不可 | `dxinputselector` の前か `dxoutputselector` の後に配置します |
| `dxgather` | 不可 | 1 つのソースから分かれたブランチを統合するもので、異なるストリーム ID を統合するものではありません |
| `videoconvert`、`videoscale`、`compositor` などの標準エレメント | 不可 | `application/x-dxvideoraw` ではなく `video/x-raw` を受け付けます |
| `dxinputselector`、`dxoutputselector` | 境界のみ | ドメインへの入口と出口になります |

誤った配置は、曖昧な実行時の結果を生むのではなく、caps ネゴシエーションの段階で失敗します。

#### 1 つの推論チェーンを共有する

すべてのチャネルが同じモデルを使う場合、単一の共有推論チェーンにすることで、チャネルごとにモデルを読み込むことを避け、NPU のメモリ使用量を削減できます。

<img src="assets/pipeline-multi-stream-single-infer.png" style="max-width: 1400px;">

現在の例は `run_multi_stream_selector.sh` を使用します。

注目ポイント: 1 つの共有チェーンを `dxinputselector name=in` と `dxoutputselector name=out` が囲んでいます。DX エレメントの数を数えて 4.7 と比較してください。

In [ ]:
if RUN_DEMOS:
    !dx_stream/pipelines/multi_stream/run_multi_stream_selector.sh
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell, or run dx_stream/pipelines/multi_stream/run_multi_stream_selector.sh in a terminal.")


In [ ]:
!sed -n '1,340p' dx_stream/pipelines/multi_stream/run_multi_stream_selector.sh


### 4.9 マルチチャネル RTSP

オプション `9` は複数の RTSP ソースを実演します。ライブパイプラインは、v3.1.x で修正されたレイテンシと QoS のレポートの恩恵を受けます。

注目ポイント: 4.8 のセレクター構成に `rtspsrc` 入力を組み合わせたもの。ストリーム URL はスクリプト先頭の変数です。お使いのカメラに合わせて編集してください。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< 9
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose 9.")


In [ ]:
!sed -n '1,300p' dx_stream/pipelines/rtsp/run_RTSP.sh


### 4.10 Secondary Mode

![Secondary Mode パイプライン](assets/pipeline-secondary.png)

- **Primary Mode(プライマリモード)** はフレーム全体を前処理して推論します。後処理は通常、新しい `DXObjectMeta` オブジェクトを作成します。
- **Secondary Mode(セカンダリモード)** は検出されたオブジェクト領域を前処理します。後処理は既存のオブジェクトメタデータを更新または拡充します。

注目ポイント: 2 組目の前処理・推論・後処理の 3 つ組に設定された `secondary-mode=true`。SCRFD が顔を検出し、その後 EfficientNet がフレーム全体ではなく各顔領域を分類します。

In [ ]:
if RUN_DEMOS:
    !./run_demo.sh <<< -
else:
    print("Skipped: set RUN_DEMOS = True in the setup cell to run this demo (it opens a window), or run ./run_demo.sh in a terminal and choose -.")


In [ ]:
!sed -n '1,320p' dx_stream/pipelines/secondary_mode/run_secondary_mode.sh


### 4.11 推論結果を MQTT または Kafka に配信する

`dxmsgconv` はカスタムのメッセージ変換ライブラリを使って推論メタデータを変換します。`dxmsgbroker` はそのペイロードを MQTT または Kafka ブローカーに配信します。

```text
... → dxpostprocess → dxmsgconv → dxmsgbroker
```

v3.1.x では、`include-frame=true` を指定すると現在のフレームが Base64 エンコードされた JPEG としてメッセージに追加されます。これにより JPEG エンコードの処理、メッセージサイズ、ネットワークトラフィックが増えるため、コンシューマーが画像を必要とする場合にのみ有効にしてください。

```bash
dxmsgconv library-file-path=/path/to/libmessage_convert.so include-frame=true ! \
dxmsgbroker broker-name=mqtt conn-info=localhost:1883 topic=test
```

パイプラインを開始する前にブローカーが起動している必要があります。同梱スクリプトには完全な MQTT と Kafka の例が含まれていますが、対話型の `run_demo.sh` メニューには含まれていません。

In [ ]:
!gst-inspect-1.0 dxmsgconv


In [ ]:
!gst-inspect-1.0 dxmsgbroker


In [ ]:
!sed -n '1,260p' dx_stream/pipelines/broker/run_dxmsgbroker_mqtt.sh


In [ ]:
!sed -n '1,260p' dx_stream/pipelines/broker/run_dxmsgbroker_kafka.sh


## 5. 独自アプリケーションの作成

この節では、チュートリアル 03 で作成した Forklift and Worker 検出器を統合します。このモデルは YOLOv7 の出力形式を使いますが、COCO の 80 クラスではなく 2 クラスしか持たないため、後処理の設定を適応させる必要があります。

![カスタムパイプライン](assets/custom-pipeline.png)

<img src="assets/detection-goal.jpg" style="max-width: 1400px;">

### 5.1 DX-STREAM v3.1.x カスタムライブラリ API

旧バージョン向けに書かれたカスタムの前処理・後処理ライブラリは、次のように更新する必要があります。

- 循環的なバッファ参照を避けるため、`DXFrameMeta::_buf` は削除されました。
- カスタム関数の最初の引数は `GstBuffer *buf` になりました。
- オブジェクトメタデータは `dx_acquire_obj_meta_from_pool()` で作成します。
- 新しいオブジェクトは `dx_add_obj_meta_to_frame()` でアタッチします。
- Primary Mode では、後処理が結果オブジェクトを作成します。
- Secondary Mode では、後処理が `object_meta` で渡されたオブジェクトを更新します。

現在の YoloV7 ライブラリは、すでに v3.1.x の関数形式を使用しています。

```cpp
extern "C" void PostProcess(
    GstBuffer* buf,
    std::vector<dxs::DXTensor> network_output,
    DXFrameMeta* frame_meta,
    DXObjectMeta* object_meta)
{
    DXObjectMeta* result = dx_acquire_obj_meta_from_pool();

    // Decode tensors and populate result.

    dx_add_obj_meta_to_frame(frame_meta, result);
}
```

### 5.2 メタデータの階層

推論結果は、別のサイドチャネルではなく `GstBuffer` と共に流れます。

```text
GstBuffer
└── DXFrameMeta
    ├── stream ID, width, height, format, ROI
    ├── input tensors  (preprocess ID → tensors)
    ├── output tensors (inference ID → tensors)
    ├── frame-level classification or segmentation
    ├── DXObjectMeta[]
    │   ├── label, confidence, box, tracking ID
    │   ├── keypoints, features, OBB, face, segmentation
    │   └── DXUserMeta[]
    └── DXUserMeta[]
```

`DXUserMeta` を使うと、アプリケーション固有のデータをフレームやオブジェクトにアタッチできます。GStreamer がバッファをコピーまたは解放したときにメタデータが有効であり続けるよう、ユーザーメタの実装はコピー関数と解放関数の両方を提供する必要があります。

### 5.3 カスタム DXNN モデルの取得

チュートリアル 03 は `yolov7-forklift-person.dxnn` をそのワークスペースにコンパイル(またはダウンロード)しました。セルはそのファイルが存在すればコピーし、存在しなければ同じモデル(71 MB)をダウンロードします。

```bash
wget -nc https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/yolov7-forklift-person.dxnn
```

In [ ]:
T03_DXNN = T03_WORKSPACE / "output" / "yolov7-forklift-person.dxnn"

if CUSTOM_DXNN.is_file():
    print("Model already present:", CUSTOM_DXNN)
elif T03_DXNN.is_file():
    !cp "{T03_DXNN}" "{CUSTOM_DXNN}"
    print("Copied from Tutorial 03:", CUSTOM_DXNN)
else:
    !cd "{WORKSPACE_DIR}" && wget -nc "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/yolov7-forklift-person.dxnn"


### 5.4 コンパイル済みモデルの確認

後処理に手を入れる前に、モデルが何を出力するかを確認します。`dxparse -v` は入力テンソルと出力テンソルを表示します。カスタムモデルは YOLOv7 のヘッドレイアウトを維持しており、raw 検出ヘッドは COCO モデルの `3 × (5 + 80) = 255` ではなく `3 × (5 + 2) = 21` チャネル(3 つのアンカー、4 つのボックス値、objectness、2 つのクラススコア)を持ちます。`libpostprocess_yolov7.so` のデコーダーはすでにこのレイアウトを処理できます。異なるのはクラス数とクラス名だけであり、5.5 のパッチが変更するのはまさにその部分です。

In [ ]:
!dxparse -m "{CUSTOM_DXNN}" -v


### 5.5 YoloV7 後処理設定の適応

5.4 で示したとおり、出力デコーダーはすでにモデルアーキテクチャに一致しています。変更が必要なのはクラスの数と名前だけです。パッチはワークスペースに書き込まれ、5.6 で SDK ソースに適用されます。

In [ ]:
%%writefile {PATCH_FILE}
diff --git a/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp b/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp
--- a/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp
+++ b/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp
@@ -55,22 +55,12 @@ struct YoloConfig {
     // Detection thresholds (adjust based on your requirements)
     float conf_threshold = 0.25f;    // Minimum confidence for detection
     float nms_threshold = 0.4f;      // IoU threshold for NMS
-    
+
     // Number of classes in your dataset
-    int num_classes = 80;
-    
+    int num_classes = 2;
+
     // COCO dataset class names (modify for your dataset)
     std::vector<std::string> class_names = {
-        "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck",
-        "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench",
-        "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra",
-        "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
-        "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
-        "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
-        "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
-        "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
-        "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
-        "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
-        "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
+        "Forklift", "Worker"
     };
 };


### 5.6 パッチの適用

以下のセルは複数回実行しても安全です。必要なときにパッチを適用し、同じパッチがすでに存在する場合は成功と報告し、ソースに競合する変更がある場合はファイルを変更せずに停止します。既存の作業をリセットしたり破棄したりすることはありません。

In [ ]:
%%bash -s "$PATCH_FILE"
PATCH_FILE="$1"
TARGET_FILE="dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp"

if git apply --check "$PATCH_FILE" 2>/dev/null; then
    git apply --whitespace=fix "$PATCH_FILE"
    echo "Patch applied successfully: $TARGET_FILE"
elif git apply --reverse --check "$PATCH_FILE" 2>/dev/null; then
    echo "Patch is already applied. No changes were needed: $TARGET_FILE"
else
    echo "ERROR: The patch is neither applicable nor already fully applied." >&2
    echo "The target may contain conflicting or partially applied changes: $TARGET_FILE" >&2
    echo "Review it with: git diff -- $TARGET_FILE" >&2
    exit 1
fi


### 5.7 DX-STREAM の再ビルド

`build.sh` は `meson` でプラグインとカスタムライブラリを再ビルドし、設定されたプレフィックスにインストールして、第 6 節用に `pydxs` を含む `venv-dx_stream` を作成します。インストール手順と root 所有のビルドディレクトリのクリーンアップには `sudo` を使うため、チュートリアル 01 と同様に、セルは `sudo` がパスワードなしで動作する場合にのみここでビルドを実行し、そうでない場合はターミナル用のコマンドを表示します。フルリビルドには数分かかります。既存の `builddir` が別の meson バージョンで生成されていたためにインクリメンタルビルドが失敗した場合、セルは `./build.sh --clean` にフォールバックし、最初からビルドし直します。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_stream
./build.sh
```

In [ ]:
if subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    # A stale builddir from an older meson makes the incremental build fail; --clean rebuilds from scratch.
    !./build.sh || ./build.sh --clean
else:
    print("sudo needs a password on this host, so the build cannot run in a cell.")
    print("Open File > New > Terminal and run:\n")
    print(f"cd {DX_STREAM_DIR}")
    print("./build.sh")


### 5.8 推論設定の作成

`dxinfer` のプロパティは、パイプライン内で直接指定することも、JSON ファイルで指定することもできます。

| JSON フィールド | 意味 |
|---|---|
| `preprocess_id` | 同じ ID を持つ `dxpreprocess` エレメントが作成した入力テンソルを選択します |
| `inference_id` | モデル出力テンソルにラベルを付けます。`dxpostprocess` は同じ ID を使う必要があります |
| `model_path` | コンパイル済み `.dxnn` モデルへのパス。相対パスはプロセスの作業ディレクトリから解決されるため、セルは絶対パスを書き込みます |
| `backend` | `auto`、`dxrt`、またはコンパイル済みの任意バックエンドを選択します |

複数の前処理、推論、後処理エレメントを含むパイプラインでは、明示的な ID を使うことが重要です。

In [ ]:
import json

yolov7_custom = {
    "preprocess_id": 1,
    "inference_id": 1,
    "model_path": str(CUSTOM_DXNN),
    "backend": "auto",
}

with open(CUSTOM_JSON, "w", encoding="utf-8") as config_file:
    json.dump(yolov7_custom, config_file, indent=2)

print(CUSTOM_JSON)
print(json.dumps(yolov7_custom, indent=2))


### 5.9 テスト動画の取得

チュートリアル 03 で使用したのと同じクリップです。そのワークスペースに存在すればコピーし、なければダウンロード(19 MB)します。

In [ ]:
T03_VIDEO = T03_WORKSPACE / "forklift-worker.mp4"

if CUSTOM_VIDEO.is_file():
    print("Video already present:", CUSTOM_VIDEO)
elif T03_VIDEO.is_file():
    !cp "{T03_VIDEO}" "{CUSTOM_VIDEO}"
    print("Copied from Tutorial 03:", CUSTOM_VIDEO)
else:
    !cd "{WORKSPACE_DIR}" && wget -nc "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/forklift-worker.mp4"


### 5.10 カスタムパイプラインの実行

ID は次の関係を形成します。

```text
dxpreprocess(preprocess-id=1)
        ↓
dxinfer(preprocess_id=1, inference_id=1)
        ↓
dxpostprocess(inference-id=1)
```

`library-file-path` は 5.7 でインストールしたライブラリを指します。以下のパスは既定のインストールプレフィックスです。DX-STREAM を `./build.sh --prefix=<dir>` でビルドした場合は `<dir>/share/gstdxstream/lib/libpostprocess_yolov7.so` を使用してください。出力は `SINK` に送られます。デスクトップセッションではウィンドウ、それ以外では `fakesink` です。

In [ ]:
print(f"Sink: {SINK}   (HEADLESS={HEADLESS}; set HEADLESS = False in the setup cell and rerun it to open a window)\n")
!gst-launch-1.0 filesrc location="{CUSTOM_VIDEO}" ! decodebin ! \
    dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
    queue max-size-buffers=1 ! \
    dxinfer config-file-path="{CUSTOM_JSON}" ! \
    queue max-size-buffers=1 ! \
    dxpostprocess \
        inference-id=1 \
        library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolov7.so \
        function-name=PostProcess ! \
    queue max-size-buffers=1 ! \
    dxosd ! {SINK}


## 6. YOLO26s 人数カウントアプリケーションの構築

この演習では、各統合ポイントを確認しやすくするために、まず Python で人数カウントを実装します。YOLO26s の物体検出を実行し、各映像フレームにアタッチされた検出メタデータを読み取り、COCO クラス `0`(`person`)を数えてテキストオーバーレイを更新します。

この節を終えると、次のことが理解できます。

- `DXObjectMeta` がどこで定義され、作成されるか
- 検出メタデータがどのように `GstBuffer` と共に流れるか
- `pydxs` がどのように C++ のメタデータを Python に公開するか
- Python の文字列がどのように実行中の GStreamer パイプラインになるか
- パッドプローブのコールバックと表示の更新が、なぜ異なる実行コンテキストを使うのか

### 6.1 YOLO26s モデルと動画の準備

`yolo26-s_640x640.dxnn` はチュートリアル 01 の 3.3 節で共有ワークスペースにダウンロードしたモデルの 1 つなので、ここではそれを再利用します。セルは、モデルが見つからない場合にのみ DX-APP のセットアップスクリプトでダウンロードします。DX-STREAM サンプルのスノーボード動画は、複数の人物が映っているため選ばれています。

In [ ]:
if not YOLO26S_MODEL.is_file():
    # Same command as Tutorial 01 section 3.3, limited to this model.
    !cd "{DX_APP_DIR}" && bash setup.sh --models yolo26s --no-force
if not YOLO26S_MODEL.is_file():
    raise FileNotFoundError(f"Model not found: {YOLO26S_MODEL}")
if not PEOPLE_VIDEO.is_file():
    raise FileNotFoundError(f"Sample video not found: {PEOPLE_VIDEO}\nRun ./setup.sh from the DX-STREAM directory first (section 2.3).")

print(f"Model: {YOLO26S_MODEL}")
print(f"Video: {PEOPLE_VIDEO}")


### 6.2 検出メタデータを理解する

DX-STREAM は、検出データを画像そのものに書き込むことなく、ピクセルと推論結果を一緒に保持します。デコードされた 1 つの映像フレームは 1 つの `GstBuffer` で運ばれ、DX-STREAM はそのバッファに 1 つの `DXFrameMeta` をアタッチし、フレームメタデータが検出オブジェクトのリストを所有します。

```text
GstBuffer — one video frame moving through the pipeline
│
├── Video memory / pixels
│
└── DXFrameMeta — frame-level GstMeta
    ├── stream_id, width, height, format, frame_rate, ROI, ...
    │
    └── object_meta_list
        ├── DXObjectMeta #0
        │   ├── label          = 0
        │   ├── label_name     = "person"
        │   ├── confidence     = 0.92
        │   └── box            = [x1, y1, x2, y2]
        ├── DXObjectMeta #1
        └── ...
```

この関係は重要です。Python のコールバックは推論を再実行せず、画像を解析もしません。`dxpostprocess` がすでに生成したオブジェクトメタデータを読み取るだけです。

#### 6.2.1 `DXObjectMeta` が定義され、作成される場所

SDK は、`DX_STREAM_DIR` からの相対パスで次のヘッダーにこれらの構造体を定義しています。

| 用途 | SDK ソース |
|---|---|
| フレームメタデータとそのオブジェクトリスト | `gst-dxstream-plugin/metadata/gst-dxframemeta.hpp` |
| オブジェクトごとの検出メタデータ | `gst-dxstream-plugin/metadata/gst-dxobjectmeta.hpp` |
| 両構造体の Python バインディング | `bindings/python/pydxs/src/metadata_binding.cpp` |

プライマリの物体検出では、`dxpostprocess` が設定された C++ の `PostProcess` 関数を呼び出します。この関数は次のライフサイクルで YOLO のテンソル出力をオブジェクトに変換します。

```cpp
DXObjectMeta* object = dx_acquire_obj_meta_from_pool();
object->_label       = class_id;
object->_label_name  = class_name;
object->_confidence  = score;
object->_box         = {x1, y1, x2, y2};
dx_add_obj_meta_to_frame(frame_meta, object);
```

結果は現在のフレームにアタッチされるため、下流のエレメントは同じ検出結果を参照します。

```text
dxinfer                 dxpostprocess                         downstream
tensor output ────────▶ create DXObjectMeta ────────┬──────▶ dxosd draws boxes
                                                    └──────▶ Python probe counts people
```

DX-STREAM はメタデータの寿命をバッファと共に管理します。あるエレメントが新しいバッファを作成して `DXFrameMeta` をコピーすると、そのオブジェクトメタデータもコピーされます。それでもアプリケーションコードは、Python のメタデータ参照を現在のバッファコールバックの処理中にのみ有効なものとして扱い、後で使うために保存しないでください。

#### 6.2.2 C++ のメタデータが Python でどう見えるか

`pydxs` は C++ のメンバーを Python で扱いやすいプロパティにマッピングします。

| C++ フィールド | Python プロパティ | この演習での意味 |
|---|---|---|
| `DXFrameMeta::_object_meta_list` | `frame_meta.object_meta_list` または `frame_meta` の反復処理 | 現在のフレーム内のすべての検出 |
| `DXObjectMeta::_label` | `obj_meta.label` | COCO クラス ID。`0` は `person` を意味します |
| `DXObjectMeta::_label_name` | `obj_meta.label_name` | 人間が読めるクラス名 |
| `DXObjectMeta::_confidence` | `obj_meta.confidence` | 検出の信頼度 |
| `DXObjectMeta::_box` | `obj_meta.box` | フレーム座標での `[left, top, right, bottom]` |
| `DXObjectMeta::_track_id` | `obj_meta.track_id` | トラッカー ID。このパイプラインでは割り当てられません |

コールバックは、`dxpostprocess` から出ようとしているのと同じ `Gst.Buffer` を受け取ります。

```python
buffer = info.get_buffer()
frame_meta = pydxs.dx_get_frame_meta(hash(buffer))

people_count = sum(
    1 for obj_meta in frame_meta
    if obj_meta.label == PERSON_CLASS_ID
)
```

`hash(buffer)` は、バインディングが期待するネイティブの `GstBuffer` アドレスを渡します。`pydxs` が `_object_meta_list` を Python のイテレータープロトコルで公開しているため、`frame_meta` は反復処理できます。

したがってこのカウントは、**現在のフレーム**で受理された `person` 検出ボックスの数です。Python コードは追加の信頼度しきい値を適用しません。信頼度によるフィルタリングや重複の抑制を行う場合は、オブジェクトがアタッチされる前に後処理側で行う必要があります。これはユニークな来訪者数ではありません。時間をまたいで同一性を維持するには追跡が必要です。

### 6.3 Python が GStreamer パイプラインを埋め込む仕組みを理解する

このアプリケーションは `gst-launch-1.0` をサブプロセスとして起動しません。代わりに、`pipeline_description` に GStreamer の launch 構文と同じ内容を Python のフォーマット済み文字列として保持し、`Gst.parse_launch()` がそのテキストを、パッドで接続された実際の GStreamer エレメントに変換します。

```text
Python application
│
├── pipeline_description = f''' ... '''
│       │
│       └── Gst.parse_launch(description)
│                    │
│                    ▼
│    ┌────────┐  ┌──────────┐  ┌───────┐  ┌─────────────┐  ┌───────┐
└───▶│ source │─▶│preprocess│─▶│ infer │─▶│ postprocess │─▶│  osd  │─▶ display
     └────────┘  └──────────┘  └───────┘  └──────┬──────┘  └───────┘
                                                  │
                                    source-pad BUFFER probe
                                                  │
                                                  ▼
                                    read metadata → count people
                                                  │
                                                  ▼
                                      update `textoverlay` text
```

プローブはバッファを観察するだけです。別のパイプラインブランチではなく、フレームをコピーすることもありません。

| パイプラインの構成要素 | 役割 |
|---|---|
| `urisourcebin ! decodebin` | 動画を読み込み、圧縮フレームをデコードします |
| `dxpreprocess` | フレームをリサイズしてモデル用に準備します |
| `dxinfer` | YOLO26s を実行し、その出力テンソルをアタッチします |
| `dxpostprocess name=detector_postprocess` | テンソルをデコードし、`DXObjectMeta` エントリをアタッチします |
| `dxosd` | オブジェクトメタデータを読み取り、ボックスとラベルを描画します |
| `videoconvert` | 標準のオーバーレイ/表示パス向けに映像フォーマットを準備します |
| `textoverlay name=people_overlay` | Python が制御するカウントを表示します |
| `fpsdisplaysink` | フレームを表示し、表示 FPS を測定します |

#### 6.3.1 パイプライン文字列から名前付き Python オブジェクトへ

次の 3 行がパイプラインのテキストとアプリケーションロジックを結び付けます。

```python
self.pipeline = Gst.parse_launch(pipeline_description)
self.people_overlay = self.pipeline.get_by_name("people_overlay")
postprocess = self.pipeline.get_by_name("detector_postprocess")
```

名前は launch 文字列内のプロパティに由来します。

```text
dxpostprocess name=detector_postprocess ...
textoverlay   name=people_overlay ...
```

その後、アプリケーションは後処理エレメントのソースパッドにコールバックをアタッチします。

```python
src_pad = postprocess.get_static_pad("src")
src_pad.add_probe(Gst.PadProbeType.BUFFER, self._postprocess_probe)
```

`dxpostprocess` から出力されるすべてのバッファは、`dxosd` に進む前に `_postprocess_probe` を呼び出します。`Gst.PadProbeReturn.OK` を返すと、同じバッファがそのまま下流に進みます。

#### 6.3.2 ストリーミングコールバックと GLib メインループ

GStreamer はストリーミングスレッドからパッドプローブを呼び出します。コールバックが遅いと後続のすべてのフレームが遅延するため、プローブはメタデータの取得、オブジェクトのカウント、表示更新のスケジューリングだけを行います。

```text
GStreamer streaming thread                    GLib main-loop thread
──────────────────────────                    ─────────────────────
buffer reaches postprocess.src
          │
          ▼
_postprocess_probe()
  ├── get GstBuffer
  ├── get DXFrameMeta
  ├── count label == 0
  └── GLib.idle_add(_update_overlays, count) ───────────────┐
          │                                                 ▼
          └── return OK                           _update_overlays()
                    │                               └── set text property
                    ▼
             buffer continues                     UI remains responsive
```

バスはパイプライン全体のイベントもメインループに報告します。

```text
GStreamer bus ── ERROR ──▶ print details and stop
              └─ EOS   ──▶ stop the main loop
```

`pipeline.set_state(Gst.State.PLAYING)` がデータフローを開始し、`GLib.MainLoop().run()` が Python アプリケーションを存続させて、アイドルコールバックとバスメッセージを処理できるようにします。

### 6.4 Python アプリケーションの作成

以下の完全なアプリケーションは、上で説明したパイプライン、メタデータプローブ、オーバーレイ更新、バス処理、コマンドライン引数を組み合わせたものです。

In [ ]:
%%writefile {APP_PATH}
#!/usr/bin/env python3
"""Run YOLO26s and display the number of detected people."""

from __future__ import annotations

import argparse
import signal
from pathlib import Path

import gi

gi.require_version("Gst", "1.0")
gi.require_version("GLib", "2.0")
from gi.repository import GLib, Gst

import pydxs


PERSON_CLASS_ID = 0


class PeopleCounterApplication:
    def __init__(self, video_path: Path, model_path: Path, postprocess_library: Path, display: bool = True):
        self.video_path = video_path.expanduser().resolve()
        self.model_path = model_path.expanduser().resolve()
        self.postprocess_library = postprocess_library.expanduser().resolve()

        for path in (self.video_path, self.model_path, self.postprocess_library):
            if not path.is_file():
                raise FileNotFoundError(path)

        Gst.init(None)
        self.loop = GLib.MainLoop()
        self.exit_code = 0
        self.last_requested_count = None
        self.frames = 0
        self.max_people = 0

        # Video sink: the display sink, or fakesink for headless runs (the probe and the count still run).
        pipeline_description = f"""
            urisourcebin uri="{self.video_path.as_uri()}" ! decodebin !
            dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 !
            queue max-size-buffers=1 !
            dxinfer preprocess-id=1 inference-id=1
                model-path="{self.model_path}" backend=auto !
            queue max-size-buffers=1 !
            dxpostprocess name=detector_postprocess inference-id=1
                library-file-path="{self.postprocess_library}"
                function-name=PostProcess !
            queue max-size-buffers=1 !
            dxosd ! videoconvert !
            textoverlay name=people_overlay
                text="People: 0" halignment=right valignment=top
                xpad=24 ypad=24 font-desc="Sans Bold 24"
                shaded-background=true !
            {{sink}}
        """.format(sink="fpsdisplaysink sync=true text-overlay=false" if display else "fakesink sync=false")

        self.pipeline = Gst.parse_launch(pipeline_description)
        self.people_overlay = self.pipeline.get_by_name("people_overlay")
        postprocess = self.pipeline.get_by_name("detector_postprocess")
        if self.people_overlay is None or postprocess is None:
            raise RuntimeError("Failed to create required GStreamer elements")

        src_pad = postprocess.get_static_pad("src")
        if src_pad is None:
            raise RuntimeError("dxpostprocess source pad was not found")
        src_pad.add_probe(Gst.PadProbeType.BUFFER, self._postprocess_probe)

        bus = self.pipeline.get_bus()
        bus.add_signal_watch()
        bus.connect("message", self._on_bus_message)

    def _postprocess_probe(self, _pad, info):
        buffer = info.get_buffer()
        if buffer is None:
            return Gst.PadProbeReturn.OK

        # hash(buffer) is the native GstBuffer address, which is what the pydxs binding expects.
        frame_meta = pydxs.dx_get_frame_meta(hash(buffer))
        if frame_meta is None:
            return Gst.PadProbeReturn.OK

        people_count = sum(
            1 for obj_meta in frame_meta if obj_meta.label == PERSON_CLASS_ID
        )
        self.frames += 1
        self.max_people = max(self.max_people, people_count)
        if people_count != self.last_requested_count:
            self.last_requested_count = people_count
            GLib.idle_add(self._update_overlays, people_count)

        return Gst.PadProbeReturn.OK

    def _update_overlays(self, people_count: int):
        self.people_overlay.set_property("text", f"People: {people_count}")
        return GLib.SOURCE_REMOVE

    def _on_bus_message(self, _bus, message):
        if message.type == Gst.MessageType.ERROR:
            error, debug = message.parse_error()
            print(f"GStreamer error: {error}")
            if debug:
                print(f"Debug details: {debug}")
            self.exit_code = 1
            self.loop.quit()
        elif message.type == Gst.MessageType.EOS:
            self.loop.quit()

    def run(self) -> int:
        state_result = self.pipeline.set_state(Gst.State.PLAYING)
        if state_result == Gst.StateChangeReturn.FAILURE:
            self.pipeline.set_state(Gst.State.NULL)
            raise RuntimeError("Failed to start the GStreamer pipeline")

        try:
            self.loop.run()
        finally:
            self.pipeline.set_state(Gst.State.NULL)
        print(f"Processed {self.frames} frames; most people seen in one frame: {self.max_people}")
        return self.exit_code

    def stop(self, *_args):
        self.loop.quit()


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--video", required=True, type=Path)
    parser.add_argument("--model", required=True, type=Path)
    parser.add_argument("--no-display", action="store_true", help="run without a window (fakesink)")
    parser.add_argument(
        "--postprocess-library",
        type=Path,
        default=Path(
            "/usr/local/share/gstdxstream/lib/"
            "libpostprocess_yolo26od.so"
        ),
    )
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    app = PeopleCounterApplication(
        args.video,
        args.model,
        args.postprocess_library,
        display=not args.no_display,
    )
    signal.signal(signal.SIGINT, app.stop)
    signal.signal(signal.SIGTERM, app.stop)
    return app.run()


if __name__ == "__main__":
    raise SystemExit(main())


### 6.5 DX-STREAM Python 環境の確認

アプリケーションは DX-STREAM の仮想環境を使う必要があります。`pydxs` モジュールと GStreamer の Python バインディングを提供しているのがこの環境だからです。

In [ ]:
if not DX_STREAM_PYTHON.is_file():
    raise FileNotFoundError(
        f"DX-STREAM Python environment was not found: {DX_STREAM_PYTHON}\n"
        "Build DX-STREAM first (section 5.7); build.sh creates venv-dx_stream."
    )

!"{DX_STREAM_PYTHON}" -m py_compile "{APP_PATH}"
!"{DX_STREAM_PYTHON}" -c "import gi, pydxs; print('gi and pydxs: OK')"


### 6.6 アプリケーションの実行

デスクトップセッション(`HEADLESS = False`)では、アプリケーションは YOLO のボックスと `People: N` オーバーレイを表示するウィンドウを開き、動画が終了するか停止ボタンが押されるまで実行中のままになります。ディスプレイがない場合(`HEADLESS = True`)は `--no-display` で実行されます。パイプライン、メタデータプローブ、カウントはそのまま動作し、フレームは `fakesink` に送られ、セルは EOS で要約行を表示して終了します。

`pydxs` と GStreamer の Python バインディングは DX-STREAM 環境にあるため、アプリケーションは Jupyter カーネルではなく `venv-dx_stream/bin/python` を使う必要があります。ターミナルでは次のように実行します。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_stream
./venv-dx_stream/bin/python <workspace>/yolo26_people_counter.py \
    --model <DX_ALL_SUITE_DIR>/workspace/res/models/yolo26-s_640x640.dxnn \
    --video dx_stream/samples/videos/snowboard.mp4
```

In [ ]:
!"{DX_STREAM_PYTHON}" "{APP_PATH}" --model "{YOLO26S_MODEL}" --video "{PEOPLE_VIDEO}" {DISPLAY_FLAG}


### 6.7 期待される結果と拡張ポイント

通常の YOLO のボックスとラベルは `dxosd` によって引き続き表示されます。標準の GStreamer テキストオーバーレイがアプリケーションの状態を追加します。

- 右上: `People: N`。カウントが変わるたびに更新されます。

表示される数は、そのフレームで受理された `person` ボックスの数と一致するはずです。1 人の人物が重なり合うボックスを生成する場合は、Python のオーバーレイで数を補正するのではなく、後処理側の信頼度/NMS ポリシーを変更してください。

本番の在室者数システムでは、`dxtracker` を追加し、入退場領域を定義して、フレームごとの raw 検出ではなく安定したトラック ID を数えてください。

## 7. デバッグ

### 7.1 対象を絞った GStreamer ログを使う

初期化と状態遷移の確認にはレベル 3 から始め、調査対象のエレメントに対してのみレベル 4 または 5 を有効にします。

```bash
# All DX-STREAM elements at INFO level
GST_DEBUG=dx*:3 ./run_demo.sh

# Inference and tracker details
GST_DEBUG=dxinfer:4,dxtracker:4 ./run_demo.sh

# Metadata lifecycle
GST_DEBUG=dxmeta:5 ./run_demo.sh

# Save logs to a file
GST_DEBUG=2,dxinfer:4 GST_DEBUG_FILE=/tmp/dxstream.log ./run_demo.sh
```

ログ出力はタイミングに影響するため、性能測定中は詳細ログを無効にしてください。

### 7.2 v3.1.x の失敗時の挙動を理解する

- 互換性のないリンクは caps ネゴシエーションの段階で失敗します。
- モデルファイルの欠落、NPU 初期化の失敗、カスタムライブラリのエラーは、`abort()` で終了する代わりに GStreamer エレメントのエラーとして報告されます。
- 処理エレメントは、測定した処理時間を LATENCY クエリに反映します。
- FLUSH 時に内部状態がリセットされ、シークと再生のやり直しの挙動が改善されています。
- EOS、FLUSH、状態遷移の際に、単一フレーム入力を含めてワーカースレッドがより確実にクリーンアップされます。

マルチストリームパイプラインがリンクできない場合は、まず標準の `video/x-raw` エレメントが `application/x-dxvideoraw` ドメインの内側に置かれていないかを確認してください。

### 7.3 パイプライングラフのエクスポート

`GST_DEBUG_DUMP_DOT_DIR` が設定されていると、GStreamer はパイプラインの状態遷移のたびに DOT ファイルを書き出します。Graphviz は `PAUSED_PLAYING` ファイルを実際のエレメントグラフの画像に変換します。必要であれば一度だけインストールしてください。

```bash
sudo apt install graphviz
```

次のセルはカーネル用にこの変数を設定し、3.1 のクイックスタートパイプラインを(設定に応じてヘッドレスまたはウィンドウ付きで)再度実行するため、DOT ファイルはチュートリアルのワークスペースに出力されます。同等のターミナルコマンドは次のとおりです。

```bash
GST_DEBUG_DUMP_DOT_DIR=<workspace>/dot gst-launch-1.0 filesrc location=... ! ... ! fakesink
```

In [ ]:
DOT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["GST_DEBUG_DUMP_DOT_DIR"] = str(DOT_DIR)
print(f"Sink: {SINK}   (HEADLESS={HEADLESS}; set HEADLESS = False in the setup cell and rerun it to open a window)\n")

!gst-launch-1.0 filesrc location="{VIDEO_SRC}" ! decodebin ! \
    dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
    queue max-size-buffers=1 ! \
    dxinfer preprocess-id=1 inference-id=1 model-path="{MODEL_PATH}" backend=auto ! \
    queue max-size-buffers=1 ! \
    dxpostprocess inference-id=1 library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolo26od.so function-name=PostProcess ! \
    queue max-size-buffers=1 ! \
    dxosd ! {SINK}

del os.environ["GST_DEBUG_DUMP_DOT_DIR"]


In [ ]:
dot_files = sorted(DOT_DIR.glob("*.dot"), key=lambda path: path.stat().st_mtime)
if not dot_files:
    raise FileNotFoundError("No DOT file was generated. Run the previous cell first.")
for dot_file in dot_files:
    print(dot_file.name)


In [ ]:
from IPython.display import Image, display

playing_files = sorted(DOT_DIR.glob("*PAUSED_PLAYING.dot"), key=lambda path: path.stat().st_mtime)
if not playing_files:
    raise FileNotFoundError("No PAUSED_PLAYING DOT file was found.")
latest_dot = playing_files[-1]
pipeline_png = DOT_DIR / "PAUSED_PLAYING.png"

!dot -Tpng -o "{pipeline_png}" "{latest_dot}"

display(Image(filename=str(pipeline_png)))


### 7.4 DX-STREAM v3.1.x での変更点

このチュートリアルは DX-STREAM v3.1.2 で検証されました。v3.1.x 系列では、このチュートリアルが前提とする次のランタイム動作が導入されました。

- `application/x-dxvideoraw` に基づく統一されたマルチストリームドメイン
- 共有処理チェーンにおけるストリームごとのライフサイクルとタイムラインの保持
- `dxinfer` バックエンドの抽象化と非同期 Put/Get 処理
- Base64 エンコードされた JPEG フレーム用の `dxmsgconv include-frame`
- カスタムライブラリとメタデータ API の更新(5.1 節)
- レイテンシレポート、FLUSH からの復帰、caps ネゴシエーション、エラー報告の改善

利用可能なバックエンドとエレメントのプロパティは、DX-STREAM のビルド方法によって異なります。インストールされているビルドは `gst-inspect-1.0` で確認してください。

## 8. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| DX-STREAM プラグインが見つからない | `DX-STREAM plugin not found` または `no element "dxinfer"` | JupyterLab を起動した環境で `GST_PLUGIN_PATH` が設定されていない | `./run-jupyter-lab.sh` の前に `GST_PLUGIN_PATH=/usr/local/lib/x86_64-linux-gnu/gstreamer-1.0`(`build.sh` が表示したパス)をエクスポートするか、`./build.sh` で再ビルドする |
| ウィンドウが表示されない | `Could not open display`、またはパイプラインが何も出力しない | グラフィカルセッションがない | デスクトップセッションで実行する。簡易確認には `fpsdisplaysink` を `fakesink` に置き換える |
| カメラパイプラインが失敗する | `v4l2src: Cannot open device` または caps ネゴシエーションエラー | デバイスパスが誤っているか、解像度がサポートされていない | `v4l2-ctl --list-formats-ext` を実行し、セル内のデバイスと caps を編集する |
| RTSP デモが接続できない | `Could not connect to server` | `run_RTSP.sh` のサンプル RTSP URL にお使いのネットワークから到達できない | `run_RTSP.sh` を到達可能なストリームに編集する |
| `build.sh` が止まる | セル内で `sudo` のパスワードプロンプトまたは `meson` エラー | ビルドはシステム全体にインストールするため `sudo` が必要 | ターミナル(File > New > Terminal)で `./build.sh` を実行する |
| 一部のフレームが全面緑色になる | ウィンドウまたは保存されたフレームが断続的に緑色に点滅する | 表示シンクの前の `dxconvert` が一部のデコード済みバッファを正しく処理できない | `videoconvert ! fpsdisplaysink`(ノートブックの既定)を使う。`dxconvert` は 3.3 の `dxscale`/`dxconvert` の実験にのみ使う |
| DOT グラフが生成されない | `No DOT file was generated` | パイプラインが PLAYING に達する前にデモが停止された | 動画の再生が始まってからセルを停止する |
| サンプルモデルが読み込み時に拒否される | `dxinfer` からの DX-RT バージョンまたはフォーマットのエラー | サンプルモデルは Model Zoo リリース `2_4_0` 向けにコンパイルされており、古いランタイムでは読み込めない | このチュートリアルの検証に使用した DX-RT バージョン(3.4.2)を使うか、お使いの DX-COM でモデルを再コンパイルする |
| `setup.sh` がパスワードを要求する | `jq` のインストール中に `[sudo] password for ...` | `jq` がなく、スクリプトが `apt` でインストールしようとしている | ターミナルで一度 `sudo apt install jq` を実行してからセルを再実行する |
| `import pydxs` が失敗する | `ModuleNotFoundError: No module named 'pydxs'` | DX-STREAM がまだビルドされていないか、`venv-dx_stream` ではなく Jupyter の Python が使われた | 5.7(ビルド)を実行し、セルと同様に `venv-dx_stream/bin/python` を使う |
| `build.sh` がすぐに失敗する | `Build data file ... was generated with an old version of meson` | 以前の DX-STREAM バージョンが残した `builddir` | `./build.sh --clean` を実行する(5.7 のセルは最初の試行が失敗すると自動的にこれを行う) |

## 9. まとめ

ここまでで次のことを行いました。

- エンドツーエンドの DX-STREAM 推論パイプラインを構築し、確認した
- 同梱の物体検出、PPU、顔検出、姿勢推定、セグメンテーション、追跡、マルチチャネル、RTSP、Secondary Mode の各デモを実行し、その背後にあるスクリプトを読んだ
- 推論バックエンド、`dxscale`、`dxconvert`、MQTT/Kafka メッセージブローカーのスクリプトを確認した(ブローカーのデモは稼働中のブローカーが必要で、メニューには含まれない)
- `application/x-dxvideoraw` マルチストリームドメインとそのエレメント配置ルールを学んだ
- 2 クラスの Forklift and Worker モデル向けに YOLOv7 の後処理と推論設定を適応させた
- 検出結果を `GstBuffer` から `DXFrameMeta`、`DXObjectMeta` を経て `pydxs` 経由で Python まで追跡した
- GStreamer パイプラインを Python に埋め込み、パッドプローブと GLib メインループを使ってフレームごとの人数カウントを表示した
- 対象を絞った GStreamer ログと DOT グラフを使ってパイプラインの挙動を診断した

### 9.1 完了チェックリスト

- [ ] `gst-inspect-1.0` で 13 個の DX-STREAM エレメントを確認した
- [ ] クイックスタートの YOLO26n パイプラインと、同梱デモを 3 つ以上実行した
- [ ] マルチストリームドメインの配置ルールを説明できる
- [ ] 2 クラスの Forklift and Worker 後処理にパッチを当て、再ビルドして実行した
- [ ] Python の人数カウントアプリケーションを実行し、`pydxs` 経由で `DXObjectMeta` を読み取った
- [ ] パイプラインの DOT グラフをエクスポートして開いた

> **次へ:** チュートリアル 05 に進み、DX-Compiler のワークフロー(ONNX の検証、キャリブレーション、PPU、グラフ最適化、量子化戦略)を詳しく学びましょう。